In [1]:
!pip install openpyxl


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
df_jo = pd.read_excel("JO_Liste_Medaille_20122014.xlsx", sheet_name = "Listemedaille")
df_licences = pd.read_csv("Licences_2015.csv", encoding = "latin-1", sep=",",dtype={"newcog2" : "string"})

cols_text_jo = ["Fédération","Discipline","Epreuve","NOM","Prénom","Sexe","MEDAILLE(S)","Année","Olympique/paralympique"]
for col in cols_text_jo:
    df_jo[col] = df_jo[col].astype("string").str.strip()

df_jo["Fédération"] = df_jo["Fédération"].str.title
df_jo["Fédération"] = df_jo["Fédération"].replace({
    "Hand Ball" : "Handball",
    "Basket Ball" : "Basket-Ball",
    "Taekwondo et Disciplines Associées" : "Taekwondo"
})

df_jo["Sexe"].astype("string").str.strip().replace({"H":"M"})

df_jo["Discipline"] = df_jo["Discipline"].fillna("Handisport (non précisé)")

/tmp/ipykernel_47907/1450702934.py:3: DtypeWarning: Columns (4: region) have mixed types. Specify dtype option on import or set low_memory=False.
  df_licences = pd.read_csv("Licences_2015.csv", encoding = "latin-1", sep=",",dtype={"newcog2" : "string"})


In [3]:
df_medailles_uniques = (
df_jo.drop_duplicates(
subset=["Fédération", "Discipline", "Epreuve", "Sexe", "MEDAILLE(S)", "Année", "Olympique/paralympique"]
)
.copy()
) 

In [4]:
df_licences.isna().sum()

newcog2               0
LIBELLE             323
fed_2014              0
code_region         323
region             4772
                   ... 
poph_20_29_2014     323
poph_30_44_2014     323
poph_45_59_2014     323
poph_60_74_2014     323
poph_75_2014        323
Length: 68, dtype: int64

In [5]:
len(df_licences)

929377

In [6]:
print(df_licences[df_licences["LIBELLE"].isna()]["newcog2"].value_counts())

newcog2
97611    35
97615    26
97610    24
97606    23
97608    23
97616    22
97603    19
97617    18
97604    16
97607    16
97609    15
97612    15
97614    15
97613    14
97605    13
97601    11
97602     9
97690     9
Name: count, dtype: int64[pyarrow]


In [7]:
print(df_licences[df_licences["pop_2014"].isna()]["newcog2"].value_counts())

newcog2
97611    35
97615    26
97610    24
97606    23
97608    23
97616    22
97603    19
97617    18
97604    16
97607    16
97609    15
97612    15
97614    15
97613    14
97605    13
97601    11
97602     9
97690     9
Name: count, dtype: int64[pyarrow]


In [8]:
print(df_licences[df_licences["code_region"].isna()]["newcog2"].value_counts())

newcog2
97611    35
97615    26
97610    24
97606    23
97608    23
97616    22
97603    19
97617    18
97604    16
97607    16
97609    15
97612    15
97614    15
97613    14
97605    13
97601    11
97602     9
97690     9
Name: count, dtype: int64[pyarrow]


In [9]:
print(df_licences[df_licences["region"].isna()]["newcog2"].value_counts())

newcog2
97415    77
97411    76
97416    72
97408    70
97418    69
         ..
97353     5
97361     3
55039     2
97357     2
97362     2
Name: count, Length: 127, dtype: int64[pyarrow]


In [10]:
df_licences["region"] = df_licences.groupby("code_region")["region"].transform("first")

In [11]:
df_licences["region"].isna().sum()

np.int64(4772)

In [12]:
print(df_licences[["code_region", "region"]].drop_duplicates().sort_values("code_region"))

        code_region                      region
924607          1.0                         NaN
925953          2.0                         NaN
927199          3.0                         NaN
927642          4.0                         NaN
755469         11.0               Île-de-France
140541         24.0         Centre-Val de Loire
152565         27.0     Bourgogne-Franche-Comté
98632          28.0                   Normandie
13992          32.0             Hauts-de-France
509458         41.0                         NaN
57079          44.0                   Grand Est
409797         52.0            Pays de la Loire
165209         53.0                    Bretagne
118676         75.0          Nouvelle-Aquitaine
63972          76.0                   Occitanie
0              84.0        Auvergne-Rhône-Alpes
35397          93.0  Provence-Alpes-Côte d'Azur
249854         94.0                       Corse
929054          NaN                         NaN


In [13]:
noms_régions_manquants = {1.0 : "Guadeloupe", 2.0 : "Martinique", 3.0 : "Guyane", 4.0 : "La Réunion",
41.0 : "Lorraine"}
df_licences["region"] = df_licences["region"].fillna(df_licences["code_region"].map(noms_régions_manquants))

In [14]:
print(df_licences.isna().sum())

newcog2              0
LIBELLE            323
fed_2014             0
code_region        323
region             323
                  ... 
poph_20_29_2014    323
poph_30_44_2014    323
poph_45_59_2014    323
poph_60_74_2014    323
poph_75_2014       323
Length: 68, dtype: int64


In [15]:
lignes_na = df_licences[df_licences["region"].isna()]
print(lignes_na["newcog2"].value_counts())

newcog2
97611    35
97615    26
97610    24
97606    23
97608    23
97616    22
97603    19
97617    18
97604    16
97607    16
97609    15
97612    15
97614    15
97613    14
97605    13
97601    11
97602     9
97690     9
Name: count, dtype: int64[pyarrow]


In [16]:
#les 323 valeurs manquantes sur la majeure partie des colonnes on un code commune commençant par 976,
# ce qui correspond à Mayotte ; on supprime donc les lignes d'observations sur Mayotte, car elles ne
# contiennent pas de données nécessaires à notre travail
df_licences = df_licences[~df_licences["newcog2"].str.startswith("976")].copy()

In [17]:
print(df_licences.isna().sum())

newcog2            0
LIBELLE            0
fed_2014           0
code_region        0
region             0
                  ..
poph_20_29_2014    0
poph_30_44_2014    0
poph_45_59_2014    0
poph_60_74_2014    0
poph_75_2014       0
Length: 68, dtype: int64


In [1]:
print("changement")

changement
